In [ ]:
# ---------------- Imports and LOCAL-RUN configuration ----------------
from pathlib import Path
from collections import Counter, defaultdict
import ast
import json
import math
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
)
from sklearn.calibration import CalibratedClassifierCV

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("XGBoost unavailable:", e)

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print("LightGBM unavailable:", e)

pd.set_option("display.max_columns", 200)

RANDOM_STATE = 42
N_SPLITS = 5
ID_COL = "person_id"
LABEL_COL = "label"
DAY_COL = "seq_day"
TOKEN_COL = "tokens"

# ---------------------------------------------------------------------
# PUBLIC-SAFE REPOSITORY PATHS
# ---------------------------------------------------------------------
# Place input CSV files under data/ and write derived outputs under outputs/.
# Do not commit data/ or outputs/ if they contain patient-level records.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "outputs" / "..."
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Expected public/reproducible inputs. These should be de-identified, analysis-ready event-day tables.
CONDITION_EVENT_DAY_PATH = DATA_DIR / "....csv"
CONDITION_EVENT_DAY_FALLBACKS = []

# Laboratory event-day files from the lab preprocessing workflow.
LAB_EVENT_DAY_PATH_MAIN = DATA_DIR / "....csv"
LAB_EVENT_DAY_PATH_SENS = DATA_DIR / "....csv"

# Choose lab file for this run. The sensitivity file may include CA19-9/tumor-marker states if available.
USE_TUMOR_MARKER_LAB_FILE = True
LAB_EVENT_DAY_PATH = LAB_EVENT_DAY_PATH_SENS if USE_TUMOR_MARKER_LAB_FILE else LAB_EVENT_DAY_PATH_MAIN

# Optional raw numeric lab table. Leave as None unless rebuilding lab states directly from raw measurements.
RAW_LAB_TABLE_PATH = None

# Optional clinical covariates/labels. If labels are already present in event-day tables, this remains useful for cohort audits.
CLINICAL_COVARIATE_PATH = DATA_DIR / "pdac_clinical_covariates_compact.csv"
USE_CLINICAL_COVARIATES = True

# Transition settings.
PRIMARY_TRANSITION_MODES = ["forward"]
SENSITIVITY_TRANSITION_MODES = ["bidirectional"]

# Lag/tau grid for condition-lab and lab-condition transitions.
# Labs can change quickly, so short windows are prioritized.
LAG_TAU_GRID = [
    (1, 1),
    (2, 1),
    (3, 1),
    (3, 2),
    (7, 2),
    (7, 3),
    (14, 3),
    (14, 7),
    (30, 7),
]

# Condition->condition transition grid.
# C2C trajectories usually tolerate a wider window than condition-lab physiology.
CONDITION_CONDITION_LAG_TAU_GRID = [
    (3, 1),
    (7, 2),
    (7, 3),
    (14, 3),
]

# Frozen family-specific settings used for primary combined modeling.
# These are based on prior single-family tuning: condition->condition broader trajectory,
# condition->lab short physiologic/workup window.
PRIMARY_C2C_LAG_TAU = (7, 3)
PRIMARY_CONDITION_TO_LAB_LAG_TAU = (2, 1)
PRIMARY_LAB_TO_CONDITION_LAG_TAU = (1, 1)
PRIMARY_PROXIMITY_LAG_TAU = (1, 1)
PRIMARY_LAB_LEVEL = "labaxis"

# Sensitivity settings for shared-window fusion.
COMBINED_SHARED_WINDOW_SETTINGS = {
    "shared_short_lag3_tau1": ((3, 1), (3, 1)),
    "shared_medium_lag7_tau3": ((7, 3), (7, 3)),
}

# Primary lab representation levels.
LAB_LEVELS = ["labstate", "labaxis"]

# Primary lab states for biologic transition features. Use abnormal-only for causal/trajectory-like analyses.
LAB_STATE_FILTERS = ["abnormal"]
# Convenience scalar used by remapping code below.
LAB_STATE_FILTER = LAB_STATE_FILTERS[0] if isinstance(LAB_STATE_FILTERS, (list, tuple)) else LAB_STATE_FILTERS

# The local lab files are already collapsed event-day tables. Set to event_day.
LAB_INPUT_MODE = "event_day"  # "auto", "event_day", or "raw"
INCLUDE_NORMAL_LAB_STATES = False  # keep False for primary biologic transition analysis

TOP_K_FREQ = 1000
TOP_K_TRANSITION = 2500
TOP_K_COMBINED = 3500
MIN_PATIENT_FREQ = 5
SPEC_TARGETS = [0.80, 0.85, 0.90, 0.95]

RUN_PREDICTION_HEAD_TUNING = False
RUN_TREE_HEADS = True
INNER_CV_SPLITS = 3

# Utility: choose first existing file among preferred path + fallbacks.
def choose_existing_path(preferred_path, fallback_paths=None, description="input file"):
    paths = [preferred_path] + list(fallback_paths or [])
    for p in paths:
        if p is not None and Path(p).exists():
            if Path(p) != Path(preferred_path):
                print(f"Using fallback {description}: {p}")
            return Path(p)
    print(f"No existing {description} found. Checked:")
    for p in paths:
        print("  -", p)
    # Return preferred path so downstream error message is explicit.
    return Path(preferred_path)

CONDITION_EVENT_DAY_PATH = choose_existing_path(
    CONDITION_EVENT_DAY_PATH,
    CONDITION_EVENT_DAY_FALLBACKS,
    description="condition event-day table",
)

print("PUBLIC SAFE NOTE: using repo-relative placeholder paths. Do not commit PHI or patient-level outputs.")
print("Output directory:", OUT_DIR)
print("Condition event-day file:", CONDITION_EVENT_DAY_PATH)
print("Lab event-day file:", LAB_EVENT_DAY_PATH)
print("Clinical covariate file:", CLINICAL_COVARIATE_PATH)
print("Use tumor-marker lab file:", USE_TUMOR_MARKER_LAB_FILE)


# Rolling prediction settings.
# These are prediction cutoffs relative to surgery/index date. For example, -30 means
# only events occurring on or before day -30 are visible to the model.
ROLLING_CUTOFF_DAYS = [-180, -120, -90, -60, -30, -14, -7, -3, 0]
MIN_HISTORY_DAY = -365
MAX_HISTORY_DAY = 0

# Primary rolling model uses frozen family-specific windows from prior pairwise tuning.
ROLLING_C2C_LAG_TAU = PRIMARY_C2C_LAG_TAU              # usually (7, 3)
ROLLING_C2LAB_LAG_TAU = PRIMARY_CONDITION_TO_LAB_LAG_TAU  # usually (2, 1)
ROLLING_LAB_LEVEL = PRIMARY_LAB_LEVEL                  # usually labaxis

# Optional sensitivity models can be toggled on after primary workflow is validated.
RUN_ROLLING_SENSITIVITY_MODELS = True
SAVE_PATIENT_LEVEL_ROLLING_PREDICTIONS = False
SAVE_ROLLING_FEATURE_SUMMARIES = True


# ---------------------------------------------------------------------
# V2 rolling memory-window and prediction-head sensitivity settings.
# ---------------------------------------------------------------------
# The short window is the primary biologically tuned model from pairwise analyses.
# The medium/long windows test whether Day -30 performance needs longer memory.
ROLLING_WINDOW_SETTINGS = {
    "short_frozen_lag7_3__lag2_1": {
        "c2c_lag_tau": (7, 3),
        "c2lab_lag_tau": (2, 1),
        "description": "primary short-memory pairwise-tuned model",
    },
    "medium_lag14_7__lag14_7": {
        "c2c_lag_tau": (14, 7),
        "c2lab_lag_tau": (14, 7),
        "description": "medium-memory sensitivity for earlier rolling cutoffs",
    },
    "long_lag45_14__lag45_14": {
        "c2c_lag_tau": (45, 14),
        "c2lab_lag_tau": (45, 14),
        "description": "long-memory sensitivity for Day -30 signal recovery",
    },
    "legacy_long_lag90_30__lag90_30": {
        "c2c_lag_tau": (90, 30),
        "c2lab_lag_tau": (90, 30),
        "description": "legacy-style long-memory sensitivity; interpret cautiously",
    },
}



# ---------------------------------------------------------------------
# V4 horizon-aware dynamic memory schedule.
# ---------------------------------------------------------------------
# The rationale is that earlier landmarks need broader historical memory,
# while later landmarks can emphasize short-range physiologic/workup signals.
# This is prespecified before model fitting, not selected using held-out test folds.
ROLLING_DYNAMIC_MEMORY_SCHEDULE = {
    -180: {"c2c_lag_tau": (90, 30), "c2lab_lag_tau": (45, 14), "description": "very early horizon; broad condition history and medium lab memory"},
    -120: {"c2c_lag_tau": (90, 30), "c2lab_lag_tau": (45, 14), "description": "early horizon; broad condition history and medium lab memory"},
    -90:  {"c2c_lag_tau": (90, 30), "c2lab_lag_tau": (45, 14), "description": "early horizon; broad condition history and medium lab memory"},
    -60:  {"c2c_lag_tau": (45, 14), "c2lab_lag_tau": (14, 7),  "description": "intermediate horizon; longer condition memory and medium lab memory"},
    -30:  {"c2c_lag_tau": (45, 14), "c2lab_lag_tau": (14, 7),  "description": "clinically actionable one-month horizon; recover longer-memory signal"},
    -14:  {"c2c_lag_tau": (14, 7),  "c2lab_lag_tau": (7, 3),   "description": "near-preoperative horizon; moderate condition and lab memory"},
    -7:   {"c2c_lag_tau": (7, 3),   "c2lab_lag_tau": (2, 1),   "description": "late horizon; short diagnostic trajectory and near-encounter physiology"},
    -3:   {"c2c_lag_tau": (7, 3),   "c2lab_lag_tau": (2, 1),   "description": "very late preoperative horizon; short diagnostic trajectory and near-encounter physiology"},
    0:    {"c2c_lag_tau": (7, 3),   "c2lab_lag_tau": (2, 1),   "description": "final pre-index horizon; short diagnostic trajectory and near-encounter physiology"},
}

RUN_DYNAMIC_MEMORY_SCHEDULE = True
DYNAMIC_MEMORY_BLOCK_NAME = "dynamic_memory_schedule"

# Main rolling run uses ridge across all feature blocks/windows.
ROLLING_MAIN_HEAD = "ridge"

# Optional prediction-head sensitivity is restricted to selected blocks/cutoffs to limit runtime.
RUN_ROLLING_HEAD_SENSITIVITY = True
ROLLING_HEAD_SENSITIVITY_CUTOFFS = [-30, -14, -7, -3, 0]
ROLLING_HEAD_SENSITIVITY_HEADS = ["ridge", "elasticnet_cv", "xgboost_calibrated", "lightgbm_calibrated"]
ROLLING_HEAD_SENSITIVITY_BLOCK_PATTERNS = [
    "condition_frequency",
    "static_burden_like",
    "condition_plus_labaxis_frequency",
    "condition_freq_plus_c2c_plus_condition_to_labaxis__short_frozen",
    "condition_freq_plus_c2c_plus_condition_to_labaxis__medium",
    "condition_freq_plus_c2c_plus_condition_to_labaxis__long",
    "condition_freq_plus_c2c_plus_condition_to_labaxis__dynamic_memory_schedule",
    "transition_only_c2c_plus_condition_to_labaxis__dynamic_memory_schedule",
    "stdp_enhanced_static_burden__dynamic_memory_schedule",
    "stdp_enhanced_static_burden__long",
]

# V4 manuscript-style replication settings.
# These add richer static burden-like features, transition-only blocks, and OOF mean ensembles.
RUN_MANUSCRIPT_STYLE_BLOCKS = True
RUN_OOF_MEAN_ENSEMBLES = True




In [ ]:
# ---------------- Local file audit ----------------
local_files = {
    "condition_event_day": CONDITION_EVENT_DAY_PATH,
    "lab_event_day_main": LAB_EVENT_DAY_PATH_MAIN,
    "lab_event_day_sensitivity_plus_tumor_markers": LAB_EVENT_DAY_PATH_SENS,
    "lab_event_day_used": LAB_EVENT_DAY_PATH,
    "clinical_covariates": CLINICAL_COVARIATE_PATH,
}

file_audit = []
for name, path in local_files.items():
    p = Path(path) if path is not None else None
    file_audit.append({
        "name": name,
        "path": str(p) if p is not None else None,
        "exists": bool(p.exists()) if p is not None else False,
    })
file_audit_df = pd.DataFrame(file_audit)
display(file_audit_df)

missing = file_audit_df.loc[~file_audit_df["exists"], "name"].tolist()
required_missing = [x for x in missing if x in {"condition_event_day", "lab_event_day_used"}]
if required_missing:
    raise FileNotFoundError(f"Required local files are missing: {required_missing}. Edit the config cell paths before running.")


In [ ]:
# ---------------- Robust event-day and raw-lab loaders ----------------
DAY_ALIASES = ["seq_day", "event_day", "days_from_surgery", "day", "days_to_surgery", "relative_day"]
TOKEN_ALIASES = ["tokens", "token", "condition_token", "lab_tokens", "lab_token", "feature", "event_node"]
VALUE_ALIASES = ["value_as_number", "numeric_value", "lab_value", "result_value", "value", "result_num"]
LAB_NAME_ALIASES = ["lab_name", "measurement_name", "concept_name", "measurement_concept_name", "test_name", "component_name", "variable", "loinc_name"]
LAB_CODE_ALIASES = ["loinc_code", "loinc", "concept_code", "measurement_source_value", "measurement_source_concept_code", "source_code", "lab_code"]
REF_LOW_ALIASES = ["range_low", "ref_low", "reference_low", "low_normal", "lower_limit", "range_low_value"]
REF_HIGH_ALIASES = ["range_high", "ref_high", "reference_high", "high_normal", "upper_limit", "range_high_value"]
UNIT_ALIASES = ["unit", "unit_source_value", "unit_concept_name", "unit_name"]
FLAG_ALIASES = ["range_flag", "abnormal_flag", "flag", "result_flag", "interpretation"]
BAD_NAME_REGEX = re.compile(r"(metrics|prediction|coefficient|importance|manifest|summary|audit|gap|inter_event)", re.I)


def parse_token_payload(x):
    """Return a clean list of tokens from scalar, list-like string, or delimited payload."""
    if isinstance(x, (list, tuple, set, np.ndarray)):
        vals = list(x)
    elif pd.isna(x):
        vals = []
    else:
        s = str(x).strip()
        vals = None
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                vals = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
            except Exception:
                vals = None
        if vals is None:
            if "|" in s:
                vals = s.split("|")
            elif ";" in s:
                vals = s.split(";")
            else:
                vals = [s]
    return sorted(set(str(v).strip() for v in vals if str(v).strip() and str(v).strip().lower() != "nan"))


def find_existing_or_raise(path, description):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {description}: {p}. Place a de-identified CSV at this path or edit the config cell."
        )
    return p


def _first_existing_col(df, aliases):
    lower = {str(c).lower(): c for c in df.columns}
    return next((lower[a] for a in aliases if a in lower), None)


def load_event_day_table(path, kind="condition"):
    path = find_existing_or_raise(path, f"{kind} event-day table")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    lower = {c.lower(): c for c in df.columns}

    id_col = lower.get(ID_COL, ID_COL)
    day_col = next((lower[c] for c in DAY_ALIASES if c in lower), None)
    tok_col = next((lower[c] for c in TOKEN_ALIASES if c in lower), None)

    if id_col not in df.columns or day_col is None or tok_col is None:
        raise ValueError(
            f"{kind} table must include person_id, a relative-day column, and a token column. Columns: {list(df.columns)}"
        )

    out = df.rename(columns={id_col: ID_COL, day_col: DAY_COL, tok_col: TOKEN_COL}).copy()
    out[ID_COL] = pd.to_numeric(out[ID_COL], errors="coerce")
    out[DAY_COL] = pd.to_numeric(out[DAY_COL], errors="coerce")
    out = out.dropna(subset=[ID_COL, DAY_COL]).copy()
    out[ID_COL] = out[ID_COL].astype(int)
    out[DAY_COL] = out[DAY_COL].astype(int)
    out[TOKEN_COL] = out[TOKEN_COL].apply(parse_token_payload)
    out = out[out[TOKEN_COL].map(len) > 0].copy()

    collapsed = (
        out.groupby([ID_COL, DAY_COL], as_index=False)[TOKEN_COL]
        .agg(lambda xs: sorted(set(t for sub in xs for t in sub)))
    )

    for optional in [LABEL_COL, "cohort_name"]:
        if optional in out.columns:
            collapsed = collapsed.merge(
                out[[ID_COL, optional]].dropna().drop_duplicates(ID_COL),
                on=ID_COL,
                how="left",
            )

    print(f"{kind} event-day table:", collapsed.shape, "patients:", collapsed[ID_COL].nunique())
    return collapsed


def load_clinical_covariates(path):
    p = Path(path)
    if not p.exists():
        print("No clinical covariate table found. Proceeding with labels from event-day tables.")
        return pd.DataFrame(columns=[ID_COL])
    df = pd.read_csv(p, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    print("Clinical covariates:", df.shape)
    return df


## Lab abnormality-state construction

For cross-modal condition-lab transitions, laboratory measurements are converted into categorical abnormality states before feature construction. The notebook uses local reference ranges when available and prespecified adult fallback thresholds when local ranges are missing. These thresholds are intended for feature engineering and physiologic-axis grouping, not for clinical diagnosis.

Primary transition models use abnormal lab states only. Normal lab measurements are treated as documentation/utilization information and are adjusted for separately through utilization features rather than used as primary biological transition nodes.



In [ ]:
# ---------------- Literature-grounded lab-state definitions and biologic-axis mapping ----------------
# Laboratory events are value-based states, not just measurement mentions.
# Priority: local reference range > structured high/low flag > adult fallback threshold.
# Fallback thresholds are for feature engineering/physiologic-axis grouping, not clinical diagnosis.

LAB_STATE_DEFINITIONS = {
    # variable_key: axis, fallback low/high thresholds, states to emit
    "ca19_9": {
        "axis": "tumor_marker",
        "fallback_high": 37.0,
        "high_state": "ca199_high",
        "rationale": "PDAC tumor marker; interpret cautiously with biliary obstruction/cholangitis/Lewis antigen status.",
    },
    "bilirubin_total": {
        "axis": "cholestasis_hepatobiliary",
        "fallback_high": 1.2,
        "high_state": "bilirubin_high",
        "rationale": "Cholestasis/hepatobiliary dysfunction; relevant to PDAC obstruction and CA19-9 interpretation.",
    },
    "alkaline_phosphatase": {
        "axis": "cholestasis_hepatobiliary",
        "high_state": "alk_phos_high",
        "rationale": "Cholestatic liver/biliary injury; use local reference range because intervals vary.",
    },
    "ast": {
        "axis": "liver_injury",
        "high_state": "ast_high",
        "rationale": "Hepatocellular injury/systemic illness axis; local reference range preferred.",
    },
    "alt": {
        "axis": "liver_injury",
        "high_state": "alt_high",
        "rationale": "Hepatocellular injury/systemic illness axis; local reference range preferred.",
    },
    "albumin": {
        "axis": "nutrition_inflammation",
        "fallback_low": 3.5,
        "low_state": "albumin_low",
        "rationale": "Nutritional/inflammatory burden.",
    },
    "wbc": {
        "axis": "inflammation",
        "fallback_high": 11.0,
        "high_state": "wbc_high",
        "rationale": "Leukocytosis/inflammatory or infectious stress.",
    },
    "neutrophil": {
        "axis": "inflammation",
        "high_state": "neutrophil_high",
        "rationale": "Neutrophil-predominant inflammation; local reference range preferred.",
    },
    "lymphocyte": {
        "axis": "immune_inflammation",
        "low_state": "lymphocyte_low",
        "rationale": "Lymphopenia/inflammatory-nutritional axis; local reference range preferred.",
    },
    "hemoglobin": {
        "axis": "hematologic",
        "fallback_low": 12.0,
        "low_state": "hemoglobin_low",
        "rationale": "Anemia/systemic illness burden; sex-specific or local reference ranges preferred.",
    },
    "platelet": {
        "axis": "hematologic_inflammation",
        "fallback_low": 150.0,
        "fallback_high": 450.0,
        "low_state": "platelet_low",
        "high_state": "platelet_high",
        "rationale": "Thrombocytopenia/thrombocytosis, inflammation or illness severity.",
    },
    "glucose": {
        "axis": "glycemic_metabolic",
        "fallback_high": 140.0,
        "high_state": "glucose_high",
        "rationale": "Hyperglycemia/diabetes-related metabolic dysregulation; random glucose fallback.",
    },
    "creatinine": {
        "axis": "renal",
        "fallback_high": 1.2,
        "high_state": "creatinine_high",
        "rationale": "Renal/systemic illness axis.",
    },
    "bun": {
        "axis": "renal",
        "fallback_high": 23.0,
        "high_state": "bun_high",
        "rationale": "Azotemia, dehydration, renal/catabolic systemic illness axis.",
    },
    "sodium": {
        "axis": "electrolyte",
        "fallback_low": 135.0,
        "fallback_high": 145.0,
        "low_state": "sodium_low",
        "high_state": "sodium_high",
        "rationale": "Fluid/electrolyte disturbance.",
    },
    "potassium": {
        "axis": "electrolyte",
        "fallback_low": 3.5,
        "fallback_high": 5.0,
        "low_state": "potassium_low",
        "high_state": "potassium_high",
        "rationale": "Renal function, intake, fluid shifts, medications, systemic illness.",
    },
}

# Synonyms and optional LOINC/code hints. Add site-specific names here after manual audit.
LAB_SYNONYMS = {
    "ca19_9": ["ca 19", "ca19", "ca_19", "carbohydrate antigen 19", "cancer antigen 19"],
    "bilirubin_total": ["total bilirubin", "bilirubin total", "bilirubin", "tbil", "bili total"],
    "alkaline_phosphatase": ["alkaline phosphatase", "alk phos", "alp"],
    "ast": ["aspartate aminotransferase", "ast", "sgot"],
    "alt": ["alanine aminotransferase", "alt", "sgpt"],
    "albumin": ["albumin"],
    "wbc": ["white blood", "wbc", "leukocyte"],
    "neutrophil": ["neutrophil", "neutrophils", "anc", "absolute neutrophil"],
    "lymphocyte": ["lymphocyte", "lymphocytes", "absolute lymph"],
    "hemoglobin": ["hemoglobin", "hgb", "hb"],
    "platelet": ["platelet", "plt"],
    "glucose": ["glucose", "blood sugar"],
    "creatinine": ["creatinine", "serum creatinine"],
    "bun": ["blood urea nitrogen", "bun", "urea nitrogen"],
    "sodium": ["sodium", "na"],
    "potassium": ["potassium", "k"],
}

LAB_CODE_HINTS = {
    # These are optional hints only; local concept names/reference ranges should be preferred.
    # Add institution-specific LOINC/code mappings here if needed.
}

LAB_AXIS_MAP = {k: v["axis"] for k, v in LAB_STATE_DEFINITIONS.items()}
# Additional non-primary labs accepted if already tokenized upstream.
LAB_AXIS_MAP.update({
    "bilirubin_direct": "cholestasis_hepatobiliary",
    "ggt": "cholestasis_hepatobiliary",
    "prealbumin": "nutrition_inflammation",
    "total_protein": "nutrition_inflammation",
    "crp": "inflammation",
    "esr": "inflammation",
    "hematocrit": "hematologic",
    "inr": "coagulation",
    "pt": "coagulation",
    "ptt": "coagulation",
    "egfr": "renal",
    "bicarbonate": "electrolyte_acid_base",
    "chloride": "electrolyte",
    "calcium": "electrolyte",
    "magnesium": "electrolyte",
    "phosphorus": "electrolyte",
    "hba1c": "glycemic_metabolic",
    "a1c": "glycemic_metabolic",
    "bmi": "nutrition_anthropometric",
    "weight": "nutrition_anthropometric",
})

ABNORMAL_STATES = {"high", "low", "abnormal", "worse", "rising", "falling", "critical_high", "critical_low"}
NORMAL_STATES = {"normal", "stable", "observed"}


def normalize_label(s):
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    s = re.sub(r"_+", "_", s)
    return s


def standardize_lab_variable(name=None, code=None):
    """Map raw lab name/code to a primary lab variable key."""
    code_s = normalize_label(code) if code is not None and not pd.isna(code) else ""
    name_s = normalize_label(name) if name is not None and not pd.isna(name) else ""
    combined = f"{name_s} {code_s}".strip()

    if code_s in LAB_CODE_HINTS:
        return LAB_CODE_HINTS[code_s]

    # Prefer longer/more specific synonyms first.
    for variable, synonyms in LAB_SYNONYMS.items():
        for syn in sorted(synonyms, key=len, reverse=True):
            syn_n = normalize_label(syn)
            if re.search(rf"(^|_){re.escape(syn_n)}(_|$)", combined):
                return variable
            if syn_n in combined and len(syn_n) >= 4:
                return variable
    return None


def _to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "")
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else np.nan


def flag_to_state(flag):
    if pd.isna(flag):
        return None
    s = str(flag).strip().lower()
    if s in {"h", "hi", "high", "above", "abnormally high", ">"}:
        return "high"
    if s in {"l", "lo", "low", "below", "abnormally low", "<"}:
        return "low"
    if "critical" in s and "high" in s:
        return "critical_high"
    if "critical" in s and "low" in s:
        return "critical_low"
    return None


def categorize_lab_value(variable, value, ref_low=np.nan, ref_high=np.nan, flag=None):
    """Return high/low/normal/None for a standardized variable using local range first, then fallback."""
    if variable not in LAB_STATE_DEFINITIONS:
        return None
    d = LAB_STATE_DEFINITIONS[variable]
    value = _to_float(value)
    ref_low = _to_float(ref_low)
    ref_high = _to_float(ref_high)

    # Structured flag can identify abnormality when numeric reference range is unavailable.
    flagged = flag_to_state(flag)

    if not pd.isna(value):
        if not pd.isna(ref_high) and value > ref_high:
            return "high"
        if not pd.isna(ref_low) and value < ref_low:
            return "low"
        if pd.isna(ref_high) and "fallback_high" in d and value > d["fallback_high"]:
            return "high"
        if pd.isna(ref_low) and "fallback_low" in d and value < d["fallback_low"]:
            return "low"
        if INCLUDE_NORMAL_LAB_STATES:
            return "normal"

    if flagged in {"high", "critical_high"} and "high_state" in d:
        return flagged
    if flagged in {"low", "critical_low"} and "low_state" in d:
        return flagged
    return None


def lab_state_name(variable, state):
    """Map generic high/low state to manuscript-style abnormality token."""
    d = LAB_STATE_DEFINITIONS.get(variable, {})
    state = normalize_label(state)
    if state in {"high", "critical_high"} and "high_state" in d:
        return d["high_state"] if state == "high" else d["high_state"].replace("_high", "_critical_high")
    if state in {"low", "critical_low"} and "low_state" in d:
        return d["low_state"] if state == "low" else d["low_state"].replace("_low", "_critical_low")
    if state == "normal":
        return f"{variable}_normal"
    return f"{variable}_{state}"


def make_value_based_lab_tokens(variable, state, level="labstate"):
    variable = normalize_label(variable)
    state = normalize_label(state)
    axis = LAB_AXIS_MAP.get(variable, "other_lab_axis")
    state_name = lab_state_name(variable, state)
    if level == "labstate":
        return f"labstate::{state_name}"
    if level == "labaxis":
        # Use axis plus high/low direction for biological interpretability.
        if state in {"critical_high"}:
            direction = "high"
        elif state in {"critical_low"}:
            direction = "low"
        elif state in {"high", "low", "normal"}:
            direction = state
        else:
            direction = "abnormal" if state in ABNORMAL_STATES else state
        return f"labaxis::{axis}::{direction}"
    if level == "labaxis_binary_abnormal":
        binary = "abnormal" if state in ABNORMAL_STATES else "not_abnormal"
        return f"labaxis::{axis}::{binary}"
    raise ValueError(f"Unknown lab level: {level}")


def parse_lab_token(tok):
    """Return lab_type/state from heterogeneous pre-tokenized lab strings."""
    raw = str(tok).strip()
    s = raw.lower().replace("::", ":")
    parts = [normalize_label(p) for p in s.split(":") if p.strip()]

    if len(parts) >= 3 and parts[0] in {"lab", "labtraj", "labstate", "tumor_marker"}:
        # Support labstate::albumin::low and labstate::albumin_low.
        if len(parts) >= 3:
            return parts[1], parts[2]

    cleaned = normalize_label(raw)
    for suffix in ["critical_high", "critical_low", "abnormal", "rising", "falling", "worse", "high", "low", "normal", "stable", "observed"]:
        suf = "_" + suffix
        if cleaned.endswith(suf):
            lab_type = cleaned[: -len(suf)]
            # If this is already a manuscript state, map back approximately.
            for var, d in LAB_STATE_DEFINITIONS.items():
                if d.get("high_state") == cleaned or d.get("low_state") == cleaned:
                    return var, suffix
            return lab_type, suffix

    return cleaned, "observed"


def lab_axis_for_type(lab_type):
    lab_type = normalize_label(lab_type)
    if lab_type in LAB_AXIS_MAP:
        return LAB_AXIS_MAP[lab_type]
    for key, axis in LAB_AXIS_MAP.items():
        if key in lab_type or lab_type in key:
            return axis
    return "other_lab_axis"


def make_lab_token(lab_type, state, level="labstate"):
    # If lab_type is a raw variable, use value-based token naming.
    var = standardize_lab_variable(lab_type, None) or normalize_label(lab_type)
    state = normalize_label(state)
    if var in LAB_STATE_DEFINITIONS and state in {"high", "low", "critical_high", "critical_low", "normal"}:
        return make_value_based_lab_tokens(var, state, level=level)

    axis = lab_axis_for_type(var)
    if level == "labstate":
        return f"labstate::{var}::{state}"
    if level == "labaxis":
        return f"labaxis::{axis}::{state}"
    if level == "labaxis_binary_abnormal":
        binary = "abnormal" if state in ABNORMAL_STATES else "not_abnormal"
        return f"labaxis::{axis}::{binary}"
    raise ValueError(f"Unknown lab level: {level}")


def lab_token_passes_filter(lab_type, state, lab_state_filter="abnormal"):
    state = normalize_label(state)
    if lab_state_filter in [None, "all"]:
        return True
    if lab_state_filter == "abnormal":
        return state in ABNORMAL_STATES or state in {"high", "low", "critical_high", "critical_low"}
    if lab_state_filter == "normal":
        return state in NORMAL_STATES
    if lab_state_filter == "observed_or_abnormal":
        return state in ABNORMAL_STATES or state in NORMAL_STATES
    raise ValueError(f"Unknown lab_state_filter: {lab_state_filter}")


def remap_lab_event_days(lab_days, level="labstate", lab_state_filter="abnormal"):
    rows = []
    for _, r in lab_days.iterrows():
        toks = []
        for tok in r[TOKEN_COL]:
            lt, st = parse_lab_token(tok)
            if lab_token_passes_filter(lt, st, lab_state_filter=lab_state_filter):
                toks.append(make_lab_token(lt, st, level=level))
        if toks:
            row = {ID_COL: int(r[ID_COL]), DAY_COL: int(r[DAY_COL]), TOKEN_COL: sorted(set(toks))}
            for c in [LABEL_COL, "cohort_name"]:
                if c in lab_days.columns:
                    row[c] = r[c]
            rows.append(row)
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return pd.DataFrame(columns=[ID_COL, DAY_COL, TOKEN_COL, LABEL_COL, "cohort_name"])
    out = (
        out.groupby([ID_COL, DAY_COL], as_index=False)[TOKEN_COL]
        .agg(lambda xs: sorted(set(t for sub in xs for t in sub)))
    ).merge(
        out[[ID_COL] + [c for c in [LABEL_COL, "cohort_name"] if c in out.columns]].drop_duplicates(ID_COL),
        on=ID_COL,
        how="left",
    )
    return out


def build_lab_event_days_from_raw(path):
    """Build abnormal lab event-days from raw numeric lab rows using local ranges and fallback thresholds."""
    path = find_existing_or_raise(path, "raw lab measurement table")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]

    id_col = _first_existing_col(df, [ID_COL, "patient_id", "person_id"])
    day_col = _first_existing_col(df, DAY_ALIASES)
    value_col = _first_existing_col(df, VALUE_ALIASES)
    name_col = _first_existing_col(df, LAB_NAME_ALIASES)
    code_col = _first_existing_col(df, LAB_CODE_ALIASES)
    low_col = _first_existing_col(df, REF_LOW_ALIASES)
    high_col = _first_existing_col(df, REF_HIGH_ALIASES)
    flag_col = _first_existing_col(df, FLAG_ALIASES)

    required_missing = [label for label, col in [("patient id", id_col), ("relative day", day_col), ("numeric value", value_col)] if col is None]
    if required_missing:
        raise ValueError(f"Raw lab table missing required columns: {required_missing}. Columns: {list(df.columns)}")
    if name_col is None and code_col is None:
        raise ValueError("Raw lab table needs at least one lab name or lab code column.")

    work = df.copy()
    work[ID_COL] = pd.to_numeric(work[id_col], errors="coerce")
    work[DAY_COL] = pd.to_numeric(work[day_col], errors="coerce")
    work["_value"] = work[value_col].apply(_to_float)
    work = work.dropna(subset=[ID_COL, DAY_COL, "_value"]).copy()
    work[ID_COL] = work[ID_COL].astype(int)
    work[DAY_COL] = work[DAY_COL].astype(int)

    work["_lab_variable"] = [
        standardize_lab_variable(work.at[i, name_col] if name_col else None, work.at[i, code_col] if code_col else None)
        for i in work.index
    ]

    def _row_state(r):
        return categorize_lab_value(
            r["_lab_variable"],
            r["_value"],
            r[low_col] if low_col else np.nan,
            r[high_col] if high_col else np.nan,
            r[flag_col] if flag_col else None,
        )

    work["_state"] = work.apply(_row_state, axis=1)
    before = len(work)
    work = work[work["_lab_variable"].notna() & work["_state"].notna()].copy()
    print("Raw lab rows parsed:", before, "-> value-based lab-state rows:", len(work))
    print("Patients with lab states:", work[ID_COL].nunique())

    work["_token"] = [make_value_based_lab_tokens(v, s, level="labstate") for v, s in zip(work["_lab_variable"], work["_state"])]

    out = (
        work.groupby([ID_COL, DAY_COL], as_index=False)["_token"]
        .agg(lambda xs: sorted(set(xs)))
        .rename(columns={"_token": TOKEN_COL})
    )

    for optional in [LABEL_COL, "cohort_name"]:
        if optional in work.columns:
            out = out.merge(work[[ID_COL, optional]].dropna().drop_duplicates(ID_COL), on=ID_COL, how="left")

    print("Lab event-day table from raw labs:", out.shape, "patients:", out[ID_COL].nunique())
    return out


def load_lab_input_event_days():
    """Load lab event-days directly or build them from raw labs depending on available files/config."""
    mode = LAB_INPUT_MODE.lower()
    if mode == "event_day":
        return load_event_day_table(LAB_EVENT_DAY_PATH, kind="lab")
    if mode == "raw":
        return build_lab_event_days_from_raw(RAW_LAB_TABLE_PATH)
    # auto
    if Path(LAB_EVENT_DAY_PATH).exists():
        return load_event_day_table(LAB_EVENT_DAY_PATH, kind="lab")
    return build_lab_event_days_from_raw(RAW_LAB_TABLE_PATH)


# Display the lab-state table used by this notebook.
lab_state_table = pd.DataFrame([
    {
        "lab_variable": k,
        "axis": v.get("axis"),
        "low_state": v.get("low_state"),
        "fallback_low": v.get("fallback_low"),
        "high_state": v.get("high_state"),
        "fallback_high": v.get("fallback_high"),
        "rationale": v.get("rationale"),
    }
    for k, v in LAB_STATE_DEFINITIONS.items()
])
display(lab_state_table)


In [ ]:
# ---------------- Load condition and lab event-day tables ----------------
condition_days = load_event_day_table(CONDITION_EVENT_DAY_PATH, kind="condition")
lab_days_raw = load_lab_input_event_days()
clinical_df = load_clinical_covariates(CLINICAL_COVARIATE_PATH) if USE_CLINICAL_COVARIATES else pd.DataFrame(columns=[ID_COL])

print("Condition event-days:", condition_days.shape, "patients:", condition_days[ID_COL].nunique())
print("Raw/prebuilt lab event-days:", lab_days_raw.shape, "patients:", lab_days_raw[ID_COL].nunique())
display(lab_days_raw.head())

# Build the lab representations used downstream.
# labstate keeps specific value-based abnormalities, e.g. labstate::bilirubin_high.
# labaxis maps those states to broader biology axes, e.g. labaxis::cholestasis_hepatobiliary::high.
lab_days_by_level = {}
for level in LAB_LEVELS:
    remapped = remap_lab_event_days(
        lab_days_raw,
        level=level,
        lab_state_filter=globals().get("LAB_STATE_FILTER", "abnormal"),
    )
    lab_days_by_level[level] = remapped
    print(f"Lab event-days [{level}]:", remapped.shape, "patients:", remapped[ID_COL].nunique())
    display(remapped.head())

# Fail early if no lab-state representation was created. This usually means the lab token parser
# does not recognize the tokens in the input lab event-day file, or the raw-lab abnormality mapping
# did not emit abnormal states.
if all(len(df) == 0 for df in lab_days_by_level.values()):
    print("WARNING: No remapped lab event-days were created. Inspect lab_days_raw tokens below:")
    display(lab_days_raw[[ID_COL, DAY_COL, TOKEN_COL]].head(20))


In [ ]:
# ---------------- Outcome table and cohort preservation ----------------
def build_y_df(condition_days, lab_days_by_level, clinical_df):
    parts = []
    if LABEL_COL in condition_days.columns:
        parts.append(condition_days[[ID_COL, LABEL_COL]].dropna())
    for df in lab_days_by_level.values():
        if LABEL_COL in df.columns:
            parts.append(df[[ID_COL, LABEL_COL]].dropna())
    if LABEL_COL in clinical_df.columns:
        parts.append(clinical_df[[ID_COL, LABEL_COL]].dropna())
    if not parts:
        for alt in ["early_recurrence", "recurrence_6m", "case", "outcome"]:
            if alt in clinical_df.columns:
                parts.append(clinical_df[[ID_COL, alt]].rename(columns={alt: LABEL_COL}).dropna())
                break
    if not parts:
        raise ValueError("Cannot find outcome label column in event-day or clinical tables.")

    y = pd.concat(parts, ignore_index=True)
    y[ID_COL] = pd.to_numeric(y[ID_COL], errors="coerce")
    y[LABEL_COL] = pd.to_numeric(y[LABEL_COL], errors="coerce")
    y = y.dropna(subset=[ID_COL, LABEL_COL]).copy()
    y[ID_COL] = y[ID_COL].astype(int)
    y[LABEL_COL] = y[LABEL_COL].astype(int)
    y = y.drop_duplicates(ID_COL)

    cohort_parts = []
    if "cohort_name" in condition_days.columns:
        cohort_parts.append(condition_days[[ID_COL, "cohort_name"]].dropna())
    for df in lab_days_by_level.values():
        if "cohort_name" in df.columns:
            cohort_parts.append(df[[ID_COL, "cohort_name"]].dropna())
    if "cohort_name" in clinical_df.columns:
        cohort_parts.append(clinical_df[[ID_COL, "cohort_name"]].dropna())
    if cohort_parts:
        cohort_df = pd.concat(cohort_parts).drop_duplicates(ID_COL)
        y = y.merge(cohort_df, on=ID_COL, how="left")
    else:
        y["cohort_name"] = ""
    return y

# Source cohort = patients with condition events or remapped lab events and a known label.
source_ids = set(condition_days[ID_COL].astype(int))
for df in lab_days_by_level.values():
    source_ids |= set(df[ID_COL].astype(int))

y_df = build_y_df(condition_days, lab_days_by_level, clinical_df)
y_df = y_df[y_df[ID_COL].isin(source_ids)].copy()
y_df = y_df.sort_values(ID_COL).reset_index(drop=True)
all_patient_ids = y_df[ID_COL].astype(int).tolist()

print("Outcome table:", y_df.shape, "patients:", y_df[ID_COL].nunique(), "case_rate:", y_df[LABEL_COL].mean())
display(y_df[LABEL_COL].value_counts().sort_index().rename_axis("label").reset_index(name="n_patients"))

assert y_df[ID_COL].is_unique, "y_df must have one row per patient."
assert set(all_patient_ids).issubset(source_ids), "y_df contains patients not in source event tables."

In [ ]:

# ---------------- Cohort initialization helper ----------------
def ensure_patient_cohort_initialized():
    """Ensure y_df and all_patient_ids exist before feature construction.

    This makes the notebook safer if a downstream cell is run manually before the
    outcome/cohort cell. The intended run mode is still Kernel -> Restart & Run All.
    """
    global y_df, all_patient_ids

    if "all_patient_ids" in globals() and "y_df" in globals():
        return

    if "build_y_df" not in globals():
        raise NameError(
            "build_y_df is not defined. Run the outcome table/cohort preservation cell first."
        )
    if "condition_days" not in globals() or "lab_days_by_level" not in globals() or "clinical_df" not in globals():
        raise NameError(
            "condition_days, lab_days_by_level, or clinical_df is not defined. Run the loader and lab-remapping cells first."
        )

    source_ids = set(condition_days[ID_COL].astype(int)) if len(condition_days) else set()
    for _df in lab_days_by_level.values():
        if _df is not None and len(_df):
            source_ids |= set(_df[ID_COL].astype(int))

    y_df = build_y_df(condition_days, lab_days_by_level, clinical_df)
    y_df = y_df[y_df[ID_COL].isin(source_ids)].copy()
    y_df = y_df.sort_values(ID_COL).reset_index(drop=True)
    all_patient_ids = y_df[ID_COL].astype(int).tolist()

    print("Initialized patient cohort:", len(all_patient_ids), "patients")
    assert y_df[ID_COL].is_unique, "y_df must have one row per patient."
    assert set(all_patient_ids).issubset(source_ids), "y_df contains patients not in source event tables."


In [ ]:
# ---------------- Sequence and feature builders ----------------
def table_to_sequences(event_df):
    seqs = {}
    if event_df is None or len(event_df) == 0:
        return seqs
    for pid, g in event_df.groupby(ID_COL):
        rows = []
        for _, r in g.sort_values(DAY_COL).iterrows():
            toks = sorted(set(str(t).strip() for t in r[TOKEN_COL] if str(t).strip()))
            if toks:
                rows.append((int(r[DAY_COL]), toks))
        seqs[int(pid)] = rows
    return seqs

ensure_patient_cohort_initialized()

condition_seq = table_to_sequences(condition_days)
lab_seq_by_level = {key: table_to_sequences(df) for key, df in lab_days_by_level.items()}


def build_frequency_dicts(seqs, prefix):
    rows = []
    for pid in all_patient_ids:
        d = {ID_COL: int(pid)}
        cnt = Counter()
        for day, toks in seqs.get(int(pid), []):
            for tok in toks:
                cnt[f"{prefix}::{tok}"] += 1
        for k, v in cnt.items():
            d[k] = math.log1p(min(v, 20))
        rows.append(d)
    return rows


def build_cross_transition_dicts(
    source_seqs,
    target_seqs,
    source_prefix,
    target_prefix,
    direction_name,
    max_lag,
    tau,
    mode="forward",
    include_same_day=False,
    exclude_same_token=False,
):
    """Build transition features for condition-lab, lab-condition, or condition-condition event pairs.

    mode='forward': source event must occur before target event.
    mode='bidirectional': source and target are paired if abs(day gap) <= max_lag, ignoring order.
    """
    rows = []
    for pid in all_patient_ids:
        d = {ID_COL: int(pid)}
        src_events = [(day, tok) for day, toks in source_seqs.get(int(pid), []) for tok in toks]
        tgt_events = [(day, tok) for day, toks in target_seqs.get(int(pid), []) for tok in toks]
        src_events = sorted(src_events)
        tgt_events = sorted(tgt_events)
        trans = Counter()

        if mode == "forward":
            for d1, a in src_events:
                for d2, b in tgt_events:
                    if exclude_same_token and str(a) == str(b):
                        continue
                    gap = d2 - d1
                    if gap < 0:
                        continue
                    if gap == 0 and not include_same_day:
                        continue
                    if gap > max_lag:
                        continue
                    w = math.exp(-gap / max(float(tau), 1e-9))
                    key = f"{direction_name}_forward_lag{max_lag}_tau{tau}::{source_prefix}:{a}->{target_prefix}:{b}"
                    trans[key] += w

        elif mode == "bidirectional":
            for d1, a in src_events:
                for d2, b in tgt_events:
                    if exclude_same_token and str(a) == str(b):
                        continue
                    gap = abs(d2 - d1)
                    if gap == 0 and not include_same_day:
                        # Same-day can be enabled separately. Default is false to reduce co-documentation dominance.
                        continue
                    if gap > max_lag:
                        continue
                    w = math.exp(-gap / max(float(tau), 1e-9))
                    # Symmetric condition-lab proximity key.
                    key = f"{direction_name}_bidirectional_lag{max_lag}_tau{tau}::{source_prefix}:{a}<->{target_prefix}:{b}"
                    trans[key] += w
        else:
            raise ValueError(f"Unknown mode: {mode}")

        for k, v in trans.items():
            d[k] = v
        rows.append(d)
    return rows


def merge_dicts(name, dict_lists):
    by_pid = {int(pid): {ID_COL: int(pid)} for pid in all_patient_ids}
    for dicts in dict_lists:
        for d in dicts:
            pid = int(d[ID_COL])
            if pid not in by_pid:
                continue
            for k, v in d.items():
                if k == ID_COL:
                    continue
                by_pid[pid][k] = by_pid[pid].get(k, 0.0) + float(v)
    return [by_pid[int(pid)] for pid in all_patient_ids]

condition_frequency = build_frequency_dicts(condition_seq, "condition_freq")

lab_frequency_by_level = {
    key: build_frequency_dicts(seq, f"{key}_freq")
    for key, seq in lab_seq_by_level.items()
}

print("Condition frequency patients:", len(condition_frequency))
for key, rows in lab_frequency_by_level.items():
    print(key, "frequency patients:", len(rows))

In [ ]:
# ---------------- Utilization/documentation intensity features ----------------
def build_utilization_dicts(condition_days, lab_days_by_level):
    rows = []
    # Prepare per-patient event summaries.
    cond_tmp = condition_days.copy()
    cond_tmp["n_tokens_day"] = cond_tmp[TOKEN_COL].map(len)

    lab_concat_parts = []
    for key, df in lab_days_by_level.items():
        tmp = df.copy()
        tmp["lab_level"] = key
        tmp["n_tokens_day"] = tmp[TOKEN_COL].map(len)
        lab_concat_parts.append(tmp)
    lab_tmp = pd.concat(lab_concat_parts, ignore_index=True) if lab_concat_parts else pd.DataFrame(columns=[ID_COL, DAY_COL, TOKEN_COL, "n_tokens_day"])

    for pid in all_patient_ids:
        d = {ID_COL: int(pid)}
        cg = cond_tmp[cond_tmp[ID_COL] == pid]
        lg = lab_tmp[lab_tmp[ID_COL] == pid]

        summary = {
            "condition_event_days": cg[DAY_COL].nunique(),
            "condition_total_tokens": cg["n_tokens_day"].sum() if len(cg) else 0,
            "condition_recent_30d": cg.loc[cg[DAY_COL] >= -30, DAY_COL].nunique() if len(cg) else 0,
            "condition_recent_90d": cg.loc[cg[DAY_COL] >= -90, DAY_COL].nunique() if len(cg) else 0,
            "lab_event_days": lg[DAY_COL].nunique(),
            "lab_total_tokens": lg["n_tokens_day"].sum() if len(lg) else 0,
            "lab_recent_30d": lg.loc[lg[DAY_COL] >= -30, DAY_COL].nunique() if len(lg) else 0,
            "lab_recent_90d": lg.loc[lg[DAY_COL] >= -90, DAY_COL].nunique() if len(lg) else 0,
        }
        for k, v in summary.items():
            d[f"util_log1p::{k}"] = math.log1p(float(v))
        rows.append(d)
    return rows

ensure_patient_cohort_initialized()

utilization_features = build_utilization_dicts(condition_days, lab_days_by_level)
print("Utilization feature patients:", len(utilization_features))

In [ ]:

# ---------------- Static clinical covariate feature block for manuscript-style baseline ----------------
def build_clinical_covariate_dicts(clinical_df):
    """Build a conservative static covariate feature block from the compact clinical table.

    This is used only as a manuscript-replication style sensitivity baseline. It intentionally
    excludes obvious identifiers, dates/timestamps, free-text source values, and outcome labels.
    Numeric features are median-imputed within the observed table. Low-cardinality categorical
    features are one-hot encoded. High-cardinality fields are skipped to reduce identifier risk.
    """
    if clinical_df is None or len(clinical_df) == 0 or ID_COL not in clinical_df.columns:
        return [{ID_COL: int(pid)} for pid in all_patient_ids]

    df = clinical_df.copy()
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce")
    df = df.dropna(subset=[ID_COL]).copy()
    df[ID_COL] = df[ID_COL].astype(int)
    df = df.drop_duplicates(ID_COL)

    exclude_exact = {
        ID_COL, LABEL_COL, "cohort", "cohort_name", "person_source_value", "source_value",
        "early_recurrence", "recurrence_6m", "case", "outcome",
    }
    exclude_substrings = [
        "date", "datetime", "time", "source_value", "source_concept", "person_source",
        "address", "phone", "email", "mrn", "name",
    ]

    candidate_cols = []
    for col in df.columns:
        c = str(col).lower()
        if col in exclude_exact:
            continue
        if any(x in c for x in exclude_substrings):
            continue
        candidate_cols.append(col)

    rows_by_pid = {int(pid): {ID_COL: int(pid)} for pid in all_patient_ids}
    use_cols = []

    for col in candidate_cols:
        ser = df[col]
        nonnull = ser.dropna()
        if len(nonnull) == 0:
            continue
        # Numeric feature.
        if pd.api.types.is_numeric_dtype(ser):
            nunique = nonnull.nunique()
            if nunique > 10000:
                continue
            vals = pd.to_numeric(ser, errors="coerce")
            med = float(vals.median()) if vals.notna().any() else 0.0
            for _, r in df[[ID_COL, col]].iterrows():
                pid = int(r[ID_COL])
                if pid in rows_by_pid:
                    val = pd.to_numeric(pd.Series([r[col]]), errors="coerce").iloc[0]
                    if pd.isna(val):
                        val = med
                    # winsorize gently to reduce extreme coding artifacts
                    rows_by_pid[pid][f"static_num::{col}"] = float(val)
            use_cols.append(col)
        else:
            # Low-cardinality categorical feature only.
            vals = nonnull.astype(str).str.strip()
            vc = vals.value_counts()
            if len(vc) > 20:
                continue
            keep = set(vc.index[:20])
            for _, r in df[[ID_COL, col]].iterrows():
                pid = int(r[ID_COL])
                if pid in rows_by_pid:
                    v = str(r[col]).strip()
                    if v and v != "nan" and v in keep:
                        rows_by_pid[pid][f"static_cat::{col}={v}"] = 1.0
            use_cols.append(col)

    print("Static clinical covariate columns used:", len(use_cols))
    if use_cols:
        print(use_cols[:50])
    return [rows_by_pid[int(pid)] for pid in all_patient_ids]

clinical_covariate_features = build_clinical_covariate_dicts(clinical_df if "clinical_df" in globals() else pd.DataFrame())
print("Clinical covariate feature patients:", len(clinical_covariate_features))


In [ ]:
# ---------------- Modeling helpers ----------------
def select_vocab(train_dicts, y_train, top_k=3000, min_patient_freq=5):
    pc = Counter()
    nc = Counter()
    dfc = Counter()
    npos = max(int(np.sum(y_train == 1)), 1)
    nneg = max(int(np.sum(y_train == 0)), 1)
    for d, y in zip(train_dicts, y_train):
        feats = [k for k in d if k != ID_COL]
        for f in feats:
            dfc[f] += 1
            pc[f] += int(y == 1)
            nc[f] += int(y == 0)
    scored = []
    for f, df in dfc.items():
        if df >= min_patient_freq:
            score = abs(pc[f] / npos - nc[f] / nneg) * math.sqrt(df)
            scored.append((score, df, f))
    scored.sort(reverse=True)
    return [f for _, _, f in scored[:top_k]]


def vectorize_dicts(dicts, vocab):
    idx = {f: i for i, f in enumerate(vocab)}
    rows, cols, data = [], [], []
    for r, d in enumerate(dicts):
        for k, v in d.items():
            if k == ID_COL:
                continue
            j = idx.get(k)
            if j is not None:
                rows.append(r)
                cols.append(j)
                data.append(float(v))
    return sparse.csr_matrix((data, (rows, cols)), shape=(len(dicts), len(vocab)), dtype=float)


def normalize_train_test(Xtr, Xte):
    if Xtr.shape[1] == 0:
        return Xtr, Xte
    return normalize(Xtr, norm="l2", axis=1), normalize(Xte, norm="l2", axis=1)


def make_model(family="ridge", y_train=None):
    if family == "ridge":
        return LogisticRegression(
            penalty="l2", solver="liblinear", C=1.0, max_iter=3000,
            class_weight="balanced", random_state=RANDOM_STATE,
        )
    if family == "elasticnet_cv":
        return LogisticRegressionCV(
            Cs=[0.03, 0.1, 0.3, 1.0, 3.0], penalty="elasticnet", solver="saga",
            l1_ratios=[0.05, 0.15, 0.5], cv=3, scoring="roc_auc",
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    if family == "xgboost_calibrated":
        if not HAS_XGB:
            return None
        npos = max(int(np.sum(y_train == 1)), 1)
        nneg = max(int(np.sum(y_train == 0)), 1)
        base = XGBClassifier(
            n_estimators=120, max_depth=2, learning_rate=0.03, subsample=0.85,
            colsample_bytree=0.75, reg_lambda=4.0, reg_alpha=0.5, min_child_weight=8,
            objective="binary:logistic", eval_metric="logloss", scale_pos_weight=nneg / npos,
            random_state=RANDOM_STATE, n_jobs=4,
        )
        return CalibratedClassifierCV(base, method="sigmoid", cv=3)
    if family == "lightgbm_calibrated":
        if not HAS_LGBM:
            return None
        npos = max(int(np.sum(y_train == 1)), 1)
        nneg = max(int(np.sum(y_train == 0)), 1)
        base = lgb.LGBMClassifier(
            objective="binary", n_estimators=160, learning_rate=0.03,
            num_leaves=7, max_depth=3, min_child_samples=40,
            subsample=0.85, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=5.0,
            scale_pos_weight=nneg / npos, random_state=RANDOM_STATE, n_jobs=4, verbose=-1,
        )
        return CalibratedClassifierCV(base, method="sigmoid", cv=3)
    raise ValueError(f"Unknown model family: {family}")


def threshold_at_specificity(y, prob, target):
    best = None
    for thr in np.unique(prob):
        pred = (prob >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        spec = tn / max(tn + fp, 1)
        if spec >= target:
            sens = tp / max(tp + fn, 1)
            ppv = tp / max(tp + fp, 1) if (tp + fp) > 0 else 0.0
            cand = (thr, sens, spec, ppv)
            if best is None or cand[1] > best[1]:
                best = cand
    if best is None:
        return np.nan, np.nan, np.nan, np.nan
    return best


def compute_metrics(y, prob, name):
    m = {
        "model": name,
        "n": int(len(y)),
        "case_rate": float(np.mean(y)),
        "auroc": float(roc_auc_score(y, prob)),
        "auprc": float(average_precision_score(y, prob)),
        "brier": float(brier_score_loss(y, prob)),
    }
    for target in SPEC_TARGETS:
        thr, sens, spec, ppv = threshold_at_specificity(y, prob, target)
        tag = f"spec{int(target * 100)}"
        m[f"{tag}_threshold"] = thr
        m[f"{tag}_sensitivity"] = sens
        m[f"{tag}_specificity"] = spec
        m[f"{tag}_ppv"] = ppv
    return m


def run_feature_model(name, dicts, family="ridge", top_k=3000):
    ids = y_df[ID_COL].astype(int).values
    y = y_df[LABEL_COL].astype(int).values
    by = {int(d[ID_COL]): d for d in dicts}
    aligned = [by.get(int(pid), {ID_COL: int(pid)}) for pid in ids]

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.full(len(ids), np.nan)
    coef_rows = []
    fold_rows = []

    for fold, (tr, te) in enumerate(skf.split(ids, y), 1):
        vocab = select_vocab([aligned[i] for i in tr], y[tr], top_k=top_k, min_patient_freq=MIN_PATIENT_FREQ)
        Xtr = vectorize_dicts([aligned[i] for i in tr], vocab)
        Xte = vectorize_dicts([aligned[i] for i in te], vocab)
        Xtr, Xte = normalize_train_test(Xtr, Xte)

        if Xtr.shape[1] == 0:
            oof[te] = np.mean(y[tr])
            fold_rows.append({"model": name, "fold": fold, "n_features": 0})
            continue

        model = make_model(family=family, y_train=y[tr])
        if model is None:
            oof[te] = np.mean(y[tr])
            fold_rows.append({"model": name, "fold": fold, "n_features": Xtr.shape[1], "skipped": True})
            continue

        model.fit(Xtr, y[tr])
        prob = model.predict_proba(Xte)[:, 1]
        oof[te] = prob
        fold_rows.append({"model": name, "fold": fold, "n_features": Xtr.shape[1], "fold_auroc": roc_auc_score(y[te], prob)})
        print(name, "fold", fold, "features", Xtr.shape[1], "AUROC", round(roc_auc_score(y[te], prob), 3))

        # Feature importance for linear models only. Tree importances are less direct after calibration.
        estimator = model
        if hasattr(model, "base_estimator"):
            estimator = model.base_estimator
        if hasattr(estimator, "coef_"):
            coefs = estimator.coef_.ravel()
            for f, c in zip(vocab, coefs):
                if abs(c) > 1e-12:
                    coef_rows.append({"model": name, "fold": fold, "feature": f, "coef": float(c), "abs_coef": float(abs(c))})

    metrics = compute_metrics(y, oof, name)
    metrics["prediction_head"] = family
    metrics["n_features_median_fold"] = float(np.median([r.get("n_features", 0) for r in fold_rows]))

    pred_df = pd.DataFrame({ID_COL: ids, LABEL_COL: y, f"p_{name}": oof})
    return metrics, pred_df, pd.DataFrame(coef_rows), pd.DataFrame(fold_rows)

In [ ]:

# ---------------- Rolling timeline truncation and V4 feature construction ----------------

def filter_event_days_to_cutoff(event_df, cutoff_day, min_history_day=MIN_HISTORY_DAY):
    """Return events visible at a rolling cutoff.

    Assumes DAY_COL is relative to surgery/index date, with negative days before index.
    At cutoff -30, only events with MIN_HISTORY_DAY <= day <= -30 are visible.
    At cutoff 0, all pre-index/index-day events are visible.
    """
    if event_df is None or len(event_df) == 0:
        return event_df
    df = event_df.copy()
    return df[(df[DAY_COL] >= min_history_day) & (df[DAY_COL] <= cutoff_day)].copy()


def get_dynamic_memory_config(cutoff_day):
    """Return prespecified horizon-aware lag/tau settings for this cutoff.

    This does not use outcome data or test-fold performance. If a cutoff is not
    explicitly listed, the nearest earlier schedule is used as a conservative fallback.
    """
    cutoff_day = int(cutoff_day)
    if cutoff_day in ROLLING_DYNAMIC_MEMORY_SCHEDULE:
        return ROLLING_DYNAMIC_MEMORY_SCHEDULE[cutoff_day]
    available = sorted(ROLLING_DYNAMIC_MEMORY_SCHEDULE)
    # Choose the closest configured cutoff by absolute distance.
    nearest = min(available, key=lambda x: abs(x - cutoff_day))
    return ROLLING_DYNAMIC_MEMORY_SCHEDULE[nearest]


def summarize_visible_events(condition_visible, lab_visible_by_level, cutoff_day):
    rows = []
    cond_counts = condition_visible.groupby(ID_COL).size() if len(condition_visible) else pd.Series(dtype=float)
    row = {
        "cutoff_day": int(cutoff_day),
        "n_patients": int(len(all_patient_ids)),
        "condition_event_days_total": int(len(condition_visible)),
        "condition_patients_with_events": int(cond_counts.shape[0]),
        "condition_event_days_per_patient_mean": float(cond_counts.reindex(all_patient_ids, fill_value=0).mean()),
    }
    for level, df in lab_visible_by_level.items():
        lab_counts = df.groupby(ID_COL).size() if len(df) else pd.Series(dtype=float)
        row[f"{level}_event_days_total"] = int(len(df))
        row[f"{level}_patients_with_events"] = int(lab_counts.shape[0])
        row[f"{level}_event_days_per_patient_mean"] = float(lab_counts.reindex(all_patient_ids, fill_value=0).mean())
    rows.append(row)
    return rows


def build_single_window_transition_blocks(condition_visible_seq, lab_visible_seq, condition_freq, lab_freq, visible_util,
                                          lab_level, setting_name, c2c_lag_tau, c2lab_lag_tau):
    """Build pairwise rolling blocks for one memory-window setting."""
    c2c_lag, c2c_tau = c2c_lag_tau
    c2lab_lag, c2lab_tau = c2lab_lag_tau

    c2c = build_cross_transition_dicts(
        condition_visible_seq,
        condition_visible_seq,
        "condition",
        "condition",
        "condition_to_condition",
        max_lag=c2c_lag,
        tau=c2c_tau,
        mode="forward",
        include_same_day=False,
        exclude_same_token=True,
    )

    c2lab = build_cross_transition_dicts(
        condition_visible_seq,
        lab_visible_seq,
        "condition",
        lab_level,
        f"condition_to_{lab_level}",
        max_lag=c2lab_lag,
        tau=c2lab_tau,
        mode="forward",
        include_same_day=False,
        exclude_same_token=False,
    )

    tag = setting_name
    blocks = {
        f"condition_freq_plus_c2c__{tag}": (
            merge_dicts("tmp", [condition_freq, c2c]), TOP_K_COMBINED
        ),
        f"condition_freq_plus_condition_to_{lab_level}__{tag}": (
            merge_dicts("tmp", [condition_freq, c2lab]), TOP_K_COMBINED
        ),
        f"condition_freq_plus_c2c_plus_condition_to_{lab_level}__{tag}": (
            merge_dicts("tmp", [condition_freq, c2c, c2lab]), TOP_K_COMBINED
        ),
    }

    if RUN_MANUSCRIPT_STYLE_BLOCKS:
        # Transition-only block approximates the manuscript's single STDP transition model.
        blocks[f"transition_only_c2c_plus_condition_to_{lab_level}__{tag}"] = (
            merge_dicts("tmp", [c2c, c2lab]), TOP_K_COMBINED
        )
        # STDP-enhanced static burden approximates static burden + STDP transition features.
        # The full manuscript baseline used richer static features when available; here we use
        # condition/lab frequency, utilization, and compact clinical covariates loaded locally.
        blocks[f"stdp_enhanced_static_burden__{tag}"] = (
            merge_dicts("tmp", [condition_freq, lab_freq, visible_util, clinical_covariate_features, c2c, c2lab]), TOP_K_COMBINED
        )

    if RUN_ROLLING_SENSITIVITY_MODELS:
        blocks[f"condition_lab_freq_plus_c2c_plus_condition_to_{lab_level}__{tag}"] = (
            merge_dicts("tmp", [condition_freq, lab_freq, c2c, c2lab]), TOP_K_COMBINED
        )
        blocks[f"condition_freq_plus_c2c_plus_condition_to_{lab_level}_plus_utilization__{tag}"] = (
            merge_dicts("tmp", [condition_freq, c2c, c2lab, visible_util]), TOP_K_COMBINED
        )
    return blocks


def build_rolling_feature_blocks(cutoff_day, lab_level=ROLLING_LAB_LEVEL):
    """Build feature dictionaries after truncating all event timelines to cutoff_day.

    V4 returns blocks for fixed memory-window settings and a prespecified
    horizon-aware dynamic schedule.
    """
    condition_visible = filter_event_days_to_cutoff(condition_days, cutoff_day)
    lab_visible_by_level = {
        level: filter_event_days_to_cutoff(df, cutoff_day)
        for level, df in lab_days_by_level.items()
    }
    if lab_level not in lab_visible_by_level:
        raise ValueError(f"Requested lab_level={lab_level}, available={list(lab_visible_by_level)}")

    condition_visible_seq = table_to_sequences(condition_visible)
    lab_visible_seq = table_to_sequences(lab_visible_by_level[lab_level])

    condition_freq = build_frequency_dicts(condition_visible_seq, "condition")
    lab_freq = build_frequency_dicts(lab_visible_seq, lab_level)
    visible_util = build_utilization_dicts(condition_visible, {lab_level: lab_visible_by_level[lab_level]})

    blocks = {
        "condition_frequency": (condition_freq, TOP_K_FREQ),
        f"condition_plus_{lab_level}_frequency": (
            merge_dicts("tmp", [condition_freq, lab_freq]), TOP_K_FREQ
        ),
    }

    if RUN_MANUSCRIPT_STYLE_BLOCKS:
        blocks["static_burden_like"] = (
            merge_dicts("tmp", [condition_freq, lab_freq, visible_util, clinical_covariate_features]), TOP_K_COMBINED
        )
        blocks["static_burden_like_no_clinical_covariates"] = (
            merge_dicts("tmp", [condition_freq, lab_freq, visible_util]), TOP_K_COMBINED
        )

    for setting_name, cfg in ROLLING_WINDOW_SETTINGS.items():
        blocks.update(
            build_single_window_transition_blocks(
                condition_visible_seq=condition_visible_seq,
                lab_visible_seq=lab_visible_seq,
                condition_freq=condition_freq,
                lab_freq=lab_freq,
                visible_util=visible_util,
                lab_level=lab_level,
                setting_name=setting_name,
                c2c_lag_tau=cfg["c2c_lag_tau"],
                c2lab_lag_tau=cfg["c2lab_lag_tau"],
            )
        )

    if RUN_DYNAMIC_MEMORY_SCHEDULE:
        dyn_cfg = get_dynamic_memory_config(cutoff_day)
        blocks.update(
            build_single_window_transition_blocks(
                condition_visible_seq=condition_visible_seq,
                lab_visible_seq=lab_visible_seq,
                condition_freq=condition_freq,
                lab_freq=lab_freq,
                visible_util=visible_util,
                lab_level=lab_level,
                setting_name=DYNAMIC_MEMORY_BLOCK_NAME,
                c2c_lag_tau=dyn_cfg["c2c_lag_tau"],
                c2lab_lag_tau=dyn_cfg["c2lab_lag_tau"],
            )
        )

    event_summary_rows = summarize_visible_events(condition_visible, lab_visible_by_level, cutoff_day)
    event_summary_rows[0]["dynamic_c2c_lag_tau"] = str(get_dynamic_memory_config(cutoff_day)["c2c_lag_tau"])
    event_summary_rows[0]["dynamic_c2lab_lag_tau"] = str(get_dynamic_memory_config(cutoff_day)["c2lab_lag_tau"])
    return blocks, pd.DataFrame(event_summary_rows)


# Quick audit of visible events across rolling cutoffs before modeling.
visible_event_summaries = []
for cutoff in ROLLING_CUTOFF_DAYS:
    _, summary = build_rolling_feature_blocks(cutoff, lab_level=ROLLING_LAB_LEVEL)
    visible_event_summaries.append(summary)
visible_event_summary_df = pd.concat(visible_event_summaries, ignore_index=True)
display(visible_event_summary_df)
visible_event_summary_df.to_csv(OUT_DIR / "rolling_visible_event_summary_v4.csv", index=False)


In [ ]:

# ---------------- Rolling out-of-fold prediction, V4 manuscript-replication style ----------------

def make_fixed_folds(ids, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return list(skf.split(ids, y))


def run_feature_model_with_folds(name, dicts, folds, family="ridge", top_k=3000):
    ids = y_df[ID_COL].astype(int).values
    y = y_df[LABEL_COL].astype(int).values
    by = {int(d[ID_COL]): d for d in dicts}
    aligned = [by.get(int(pid), {ID_COL: int(pid)}) for pid in ids]

    oof = np.full(len(ids), np.nan)
    fold_rows = []

    for fold, (tr, te) in enumerate(folds, 1):
        vocab = select_vocab([aligned[i] for i in tr], y[tr], top_k=top_k, min_patient_freq=MIN_PATIENT_FREQ)
        Xtr = vectorize_dicts([aligned[i] for i in tr], vocab)
        Xte = vectorize_dicts([aligned[i] for i in te], vocab)
        Xtr, Xte = normalize_train_test(Xtr, Xte)

        if Xtr.shape[1] == 0:
            oof[te] = np.mean(y[tr])
            fold_rows.append({"model": name, "fold": fold, "n_features": 0, "fallback_mean": True})
            continue

        model = make_model(family=family, y_train=y[tr])
        if model is None:
            oof[te] = np.mean(y[tr])
            fold_rows.append({"model": name, "fold": fold, "n_features": Xtr.shape[1], "skipped": True})
            continue

        model.fit(Xtr, y[tr])
        prob = model.predict_proba(Xte)[:, 1]
        oof[te] = prob
        fold_auc = roc_auc_score(y[te], prob) if len(np.unique(y[te])) == 2 else np.nan
        fold_rows.append({"model": name, "fold": fold, "n_features": Xtr.shape[1], "fold_auroc": fold_auc})

    metrics = compute_metrics(y, oof, name)
    metrics["prediction_head"] = family
    metrics["n_features_median_fold"] = float(np.median([r.get("n_features", 0) for r in fold_rows]))
    pred_df = pd.DataFrame({ID_COL: ids, LABEL_COL: y, "risk": oof})
    return metrics, pred_df, pd.DataFrame(fold_rows)


def block_matches_any_pattern(block_name, patterns):
    return any(pattern in block_name for pattern in patterns)


ids = y_df[ID_COL].astype(int).values
y = y_df[LABEL_COL].astype(int).values
fixed_folds = make_fixed_folds(ids, y)

rolling_metric_rows = []
rolling_pred_tables = []
rolling_fold_tables = []

# Cache feature blocks by cutoff so optional head sensitivity can reuse the same truncated features.
rolling_blocks_cache = {}

for cutoff in ROLLING_CUTOFF_DAYS:
    print("\n==============================")
    print("Rolling cutoff day:", cutoff)
    print("==============================")
    blocks, _summary = build_rolling_feature_blocks(cutoff, lab_level=ROLLING_LAB_LEVEL)
    rolling_blocks_cache[int(cutoff)] = blocks

    # Main V4 run: ridge across all fixed and dynamic memory-window feature blocks.
    for block_name, (dicts, top_k) in blocks.items():
        model_name = f"day{cutoff}_{block_name}_{ROLLING_MAIN_HEAD}"
        print("Running", model_name)
        metrics, pred_df, fold_df = run_feature_model_with_folds(
            model_name,
            dicts,
            fixed_folds,
            family=ROLLING_MAIN_HEAD,
            top_k=top_k,
        )
        metrics["cutoff_day"] = int(cutoff)
        metrics["feature_block"] = block_name
        metrics["run_type"] = "main_all_blocks"
        rolling_metric_rows.append(metrics)

        pred_df["cutoff_day"] = int(cutoff)
        pred_df["feature_block"] = block_name
        pred_df["model"] = model_name
        pred_df["prediction_head"] = ROLLING_MAIN_HEAD
        pred_df["run_type"] = "main_all_blocks"
        rolling_pred_tables.append(pred_df)

        fold_df["cutoff_day"] = int(cutoff)
        fold_df["feature_block"] = block_name
        fold_df["prediction_head"] = ROLLING_MAIN_HEAD
        fold_df["run_type"] = "main_all_blocks"
        rolling_fold_tables.append(fold_df)

# Optional head sensitivity: selected cutoffs and blocks only.
if RUN_ROLLING_HEAD_SENSITIVITY:
    for cutoff in ROLLING_HEAD_SENSITIVITY_CUTOFFS:
        if int(cutoff) not in rolling_blocks_cache:
            continue
        blocks = rolling_blocks_cache[int(cutoff)]
        selected_blocks = {
            block_name: value
            for block_name, value in blocks.items()
            if block_matches_any_pattern(block_name, ROLLING_HEAD_SENSITIVITY_BLOCK_PATTERNS)
        }
        print("\n==============================")
        print("Prediction-head sensitivity cutoff day:", cutoff)
        print("Selected blocks:", list(selected_blocks))
        print("==============================")
        for block_name, (dicts, top_k) in selected_blocks.items():
            for head in ROLLING_HEAD_SENSITIVITY_HEADS:
                if head == ROLLING_MAIN_HEAD:
                    # Already run above; skip duplicate main ridge fit.
                    continue
                if head == "xgboost_calibrated" and not HAS_XGB:
                    continue
                if head == "lightgbm_calibrated" and not HAS_LGBM:
                    continue
                model_name = f"day{cutoff}_{block_name}_{head}"
                print("Running", model_name)
                metrics, pred_df, fold_df = run_feature_model_with_folds(
                    model_name,
                    dicts,
                    fixed_folds,
                    family=head,
                    top_k=top_k,
                )
                metrics["cutoff_day"] = int(cutoff)
                metrics["feature_block"] = block_name
                metrics["run_type"] = "head_sensitivity_selected_blocks"
                rolling_metric_rows.append(metrics)

                pred_df["cutoff_day"] = int(cutoff)
                pred_df["feature_block"] = block_name
                pred_df["model"] = model_name
                pred_df["prediction_head"] = head
                pred_df["run_type"] = "head_sensitivity_selected_blocks"
                rolling_pred_tables.append(pred_df)

                fold_df["cutoff_day"] = int(cutoff)
                fold_df["feature_block"] = block_name
                fold_df["prediction_head"] = head
                fold_df["run_type"] = "head_sensitivity_selected_blocks"
                rolling_fold_tables.append(fold_df)



def build_oof_mean_ensembles(rolling_predictions_long, rolling_metrics_df):
    """Create prespecified equal-weight OOF ensembles from already-generated OOF predictions.

    This is a safe post-hoc ensemble in the sense that each component prediction is out-of-fold.
    It does not train a new meta-learner on all OOF predictions. It is therefore closer to an
    equal-weight calibrated/component ensemble than to a stacked model. It is not an exact
    reproduction of the manuscript calibrated STDP ensemble, but it tests whether the Day -30
    performance gap is driven by component ensembling and richer feature blocks.
    """
    if not RUN_OOF_MEAN_ENSEMBLES:
        return [], []

    ids = y_df[ID_COL].astype(int).values
    y = y_df[LABEL_COL].astype(int).values
    ensemble_metric_rows = []
    ensemble_pred_tables = []

    ensemble_definitions = [
        {
            "name": "manuscript_style_equal_weight_ensemble",
            "include_feature_substrings": [
                "static_burden_like",
                "transition_only_c2c_plus_condition_to_labaxis__dynamic_memory_schedule",
                "stdp_enhanced_static_burden__dynamic_memory_schedule",
                "condition_freq_plus_c2c_plus_condition_to_labaxis__dynamic_memory_schedule",
            ],
            "include_heads": ["ridge", "elasticnet_cv", "xgboost_calibrated", "lightgbm_calibrated"],
        },
        {
            "name": "day30_long_memory_equal_weight_ensemble",
            "include_feature_substrings": [
                "static_burden_like",
                "stdp_enhanced_static_burden__long",
                "condition_freq_plus_c2c_plus_condition_to_labaxis__long",
                "condition_freq_plus_c2c_plus_condition_to_labaxis__legacy_long",
            ],
            "include_heads": ["ridge", "elasticnet_cv", "xgboost_calibrated", "lightgbm_calibrated"],
        },
        {
            "name": "linear_only_equal_weight_ensemble",
            "include_feature_substrings": [
                "static_burden_like",
                "transition_only_c2c_plus_condition_to_labaxis__dynamic_memory_schedule",
                "stdp_enhanced_static_burden__dynamic_memory_schedule",
                "condition_freq_plus_c2c_plus_condition_to_labaxis__dynamic_memory_schedule",
            ],
            "include_heads": ["ridge", "elasticnet_cv"],
        },
    ]

    for cutoff in sorted(rolling_predictions_long["cutoff_day"].unique()):
        cut = rolling_predictions_long[rolling_predictions_long["cutoff_day"] == cutoff].copy()
        for ens in ensemble_definitions:
            keep_models = []
            for model_name, g in cut.groupby("model"):
                fb = str(g["feature_block"].iloc[0])
                head = str(g["prediction_head"].iloc[0])
                if head not in ens["include_heads"]:
                    continue
                if any(pat in fb for pat in ens["include_feature_substrings"]):
                    keep_models.append(model_name)
            keep_models = sorted(set(keep_models))
            if len(keep_models) < 2:
                continue

            wide = None
            for model_name in keep_models:
                tmp = cut.loc[cut["model"] == model_name, [ID_COL, LABEL_COL, "risk"]].rename(columns={"risk": model_name})
                wide = tmp if wide is None else wide.merge(tmp, on=[ID_COL, LABEL_COL], how="inner")
            if wide is None or wide.shape[1] <= 3:
                continue
            model_cols = [c for c in wide.columns if c not in [ID_COL, LABEL_COL]]
            wide["risk"] = wide[model_cols].mean(axis=1)
            # Align order to y_df.
            wide = y_df[[ID_COL, LABEL_COL]].merge(wide[[ID_COL, "risk"]], on=ID_COL, how="left")
            if wide["risk"].isna().any():
                continue
            name = f"day{int(cutoff)}_{ens['name']}"
            metrics = compute_metrics(y, wide["risk"].values, name)
            metrics["prediction_head"] = "oof_equal_weight_mean"
            metrics["n_features_median_fold"] = np.nan
            metrics["cutoff_day"] = int(cutoff)
            metrics["feature_block"] = ens["name"]
            metrics["run_type"] = "oof_mean_ensemble_prespecified"
            metrics["ensemble_n_components"] = len(model_cols)
            metrics["ensemble_components"] = " | ".join(model_cols[:20])
            ensemble_metric_rows.append(metrics)

            pred_df = pd.DataFrame({ID_COL: ids, LABEL_COL: y, "risk": wide["risk"].values})
            pred_df["cutoff_day"] = int(cutoff)
            pred_df["feature_block"] = ens["name"]
            pred_df["model"] = name
            pred_df["prediction_head"] = "oof_equal_weight_mean"
            pred_df["run_type"] = "oof_mean_ensemble_prespecified"
            ensemble_pred_tables.append(pred_df)
            print("Built ensemble", name, "components", len(model_cols), "AUROC", round(metrics["auroc"], 3))
    return ensemble_metric_rows, ensemble_pred_tables


rolling_metrics_df = pd.DataFrame(rolling_metric_rows).sort_values(["cutoff_day", "auroc"], ascending=[True, False])
rolling_predictions_long = pd.concat(rolling_pred_tables, ignore_index=True)
rolling_fold_df = pd.concat(rolling_fold_tables, ignore_index=True)

ensemble_metric_rows, ensemble_pred_tables = build_oof_mean_ensembles(rolling_predictions_long, rolling_metrics_df)
if ensemble_metric_rows:
    rolling_metrics_df = pd.concat([rolling_metrics_df, pd.DataFrame(ensemble_metric_rows)], ignore_index=True)
    rolling_metrics_df = rolling_metrics_df.sort_values(["cutoff_day", "auroc"], ascending=[True, False])
if ensemble_pred_tables:
    rolling_predictions_long = pd.concat([rolling_predictions_long] + ensemble_pred_tables, ignore_index=True)

rolling_metrics_df.to_csv(OUT_DIR / "rolling_model_metrics_by_cutoff_v4.csv", index=False)
rolling_fold_df.to_csv(OUT_DIR / "rolling_model_fold_summaries_v4.csv", index=False)
if SAVE_PATIENT_LEVEL_ROLLING_PREDICTIONS:
    rolling_predictions_long.to_csv(OUT_DIR / "rolling_patient_level_oof_predictions_LONG_private_v4.csv", index=False)

# Convenience tables.
best_by_cutoff = (
    rolling_metrics_df.sort_values(["cutoff_day", "auroc"], ascending=[True, False])
    .groupby("cutoff_day")
    .head(5)
    .reset_index(drop=True)
)
best_by_cutoff.to_csv(OUT_DIR / "rolling_top5_models_by_cutoff_v4.csv", index=False)

day30_table = rolling_metrics_df[rolling_metrics_df["cutoff_day"] == -30].sort_values("auroc", ascending=False)
day30_table.to_csv(OUT_DIR / "rolling_day_minus_30_model_comparison_v4.csv", index=False)

print("Top Day -30 models:")
display(day30_table.head(20))
print("Top models by cutoff:")
display(best_by_cutoff)


# Manuscript-replication Day -30 focus table.
replication_patterns = [
    "static_burden_like",
    "transition_only",
    "stdp_enhanced_static_burden",
    "manuscript_style_equal_weight_ensemble",
    "day30_long_memory_equal_weight_ensemble",
    "linear_only_equal_weight_ensemble",
]
replication_day30 = day30_table[
    day30_table["feature_block"].astype(str).apply(lambda x: any(p in x for p in replication_patterns))
].copy()
replication_day30.to_csv(OUT_DIR / "rolling_day_minus_30_manuscript_replication_focus_v4.csv", index=False)
print("Manuscript-replication style Day -30 candidates:")
display(replication_day30.head(30))


In [ ]:

# ---------------- Rolling performance plots ----------------
primary_block = "condition_freq_plus_c2c_plus_condition_to_labaxis__short_frozen_lag7_3__lag2_1"
medium_block = "condition_freq_plus_c2c_plus_condition_to_labaxis__medium_lag14_7__lag14_7"
long_block = "condition_freq_plus_c2c_plus_condition_to_labaxis__long_lag45_14__lag45_14"
legacy_long_block = "condition_freq_plus_c2c_plus_condition_to_labaxis__legacy_long_lag90_30__lag90_30"
dynamic_block = "condition_freq_plus_c2c_plus_condition_to_labaxis__dynamic_memory_schedule"

plot_df = rolling_metrics_df[rolling_metrics_df["prediction_head"] == ROLLING_MAIN_HEAD].copy()

highlight_blocks = {
    "condition_frequency",
    primary_block,
    medium_block,
    long_block,
    legacy_long_block,
    dynamic_block,
}

for metric in ["auroc", "auprc", "brier"]:
    plt.figure(figsize=(10, 5.5))
    for block, g in plot_df.groupby("feature_block"):
        if block not in highlight_blocks:
            continue
        g = g.sort_values("cutoff_day")
        lw = 3.0 if block in {primary_block, medium_block, long_block, dynamic_block} else 1.8
        alpha = 1.0 if block in {primary_block, medium_block, long_block, dynamic_block} else 0.75
        plt.plot(g["cutoff_day"], g[metric], marker="o", linewidth=lw, alpha=alpha, label=block)
    plt.axvline(0, linestyle="--", linewidth=1)
    plt.xlabel("Prediction cutoff day relative to surgery/index")
    plt.ylabel(metric.upper())
    plt.title(f"Rolling {metric.upper()} by memory-window setting")
    plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"rolling_v3_{metric}_by_memory_window.png", dpi=300, bbox_inches="tight")
    plt.show()

# Head sensitivity at Day -30 and later.
if RUN_ROLLING_HEAD_SENSITIVITY:
    head_plot = rolling_metrics_df[
        (rolling_metrics_df["cutoff_day"].isin(ROLLING_HEAD_SENSITIVITY_CUTOFFS))
        & (rolling_metrics_df["feature_block"].isin(["condition_frequency", primary_block, medium_block, long_block, dynamic_block]))
    ].copy()
    display(head_plot.sort_values(["cutoff_day", "auroc"], ascending=[True, False]))
    head_plot.to_csv(OUT_DIR / "rolling_v3_head_sensitivity_selected_blocks.csv", index=False)

# Case/control mean predicted risk trajectories for the V4 dynamic-memory combined model.
primary_pred = rolling_predictions_long[
    (rolling_predictions_long["feature_block"] == primary_block)
    & (rolling_predictions_long["prediction_head"] == ROLLING_MAIN_HEAD)
].copy()
mean_risk = (
    primary_pred.groupby(["cutoff_day", LABEL_COL])["risk"]
    .agg(["mean", "median", "count"])
    .reset_index()
)
display(mean_risk)
mean_risk.to_csv(OUT_DIR / "rolling_v3_primary_mean_risk_by_label.csv", index=False)

plt.figure(figsize=(8, 5))
for lab, g in mean_risk.groupby(LABEL_COL):
    g = g.sort_values("cutoff_day")
    label = "recurrence" if int(lab) == 1 else "control"
    plt.plot(g["cutoff_day"], g["mean"], marker="o", linewidth=2.5, label=label)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Prediction cutoff day relative to surgery/index")
plt.ylabel("Mean out-of-fold predicted risk")
plt.title("Rolling mean predicted risk by outcome group, dynamic-memory model")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "rolling_v3_primary_mean_risk_by_outcome.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# ---------------- Patient-level risk trajectory summaries ----------------
# Wide table: one row per patient for the primary rolling model.
primary_wide = primary_pred.pivot_table(
    index=[ID_COL, LABEL_COL],
    columns="cutoff_day",
    values="risk",
    aggfunc="first",
).reset_index()
primary_wide.columns = [str(c) if not isinstance(c, tuple) else str(c) for c in primary_wide.columns]

# Risk deltas between clinically relevant windows.
def add_delta(df, start_day, end_day):
    s = str(start_day)
    e = str(end_day)
    if s in df.columns and e in df.columns:
        df[f"delta_{start_day}_to_{end_day}"] = df[e] - df[s]

for start_day, end_day in [(-180, -30), (-90, -30), (-30, 0), (-14, 0), (-7, 0), (-180, 0)]:
    add_delta(primary_wide, start_day, end_day)

if SAVE_PATIENT_LEVEL_ROLLING_PREDICTIONS:
    primary_wide.to_csv(OUT_DIR / "rolling_primary_patient_risk_trajectories_WIDE_private.csv", index=False)

display(primary_wide.head())

# Delta-risk summary by outcome.
delta_cols = [c for c in primary_wide.columns if c.startswith("delta_")]
if delta_cols:
    delta_summary = primary_wide.groupby(LABEL_COL)[delta_cols].agg(["mean", "median", "std", "count"])
    display(delta_summary)
    delta_summary.to_csv(OUT_DIR / "rolling_primary_delta_risk_summary_by_label.csv")

# Plot a small anonymized sample of patient trajectories.
rng = np.random.default_rng(RANDOM_STATE)
sample_case_ids = primary_wide.loc[primary_wide[LABEL_COL] == 1, ID_COL].sample(
    n=min(20, int((primary_wide[LABEL_COL] == 1).sum())), random_state=RANDOM_STATE
).tolist()
sample_control_ids = primary_wide.loc[primary_wide[LABEL_COL] == 0, ID_COL].sample(
    n=min(20, int((primary_wide[LABEL_COL] == 0).sum())), random_state=RANDOM_STATE
).tolist()

plt.figure(figsize=(9, 5))
for pid in sample_control_ids:
    g = primary_pred[primary_pred[ID_COL] == pid].sort_values("cutoff_day")
    plt.plot(g["cutoff_day"], g["risk"], linewidth=0.8, alpha=0.35)
for pid in sample_case_ids:
    g = primary_pred[primary_pred[ID_COL] == pid].sort_values("cutoff_day")
    plt.plot(g["cutoff_day"], g["risk"], linewidth=1.2, alpha=0.65)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Prediction cutoff day relative to surgery/index")
plt.ylabel("Out-of-fold predicted risk")
plt.title("Example rolling risk trajectories, recurrence cases overlaid on controls")
plt.tight_layout()
plt.savefig(OUT_DIR / "rolling_primary_example_patient_trajectories_private.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# ---------------- Rolling VVUQ manifest ----------------
manifest = {
    "notebook": "rolling_risk_condition_condition_lab_stdp_v3_dynamic_memory_schedule",
    "task": "simulate accumulating pre-index EHR timelines for recurrence risk prediction",
    "id_col": ID_COL,
    "label_col": LABEL_COL,
    "n_patients": int(len(y_df)),
    "case_rate": float(y_df[LABEL_COL].mean()),
    "rolling_cutoff_days": ROLLING_CUTOFF_DAYS,
    "min_history_day": MIN_HISTORY_DAY,
    "primary_feature_block": dynamic_block,
    "short_memory_reference_block": primary_block,
    "primary_c2c_lag_tau": PRIMARY_C2C_LAG_TAU,
    "primary_condition_to_lab_lag_tau": PRIMARY_CONDITION_TO_LAB_LAG_TAU,
    "rolling_window_settings": ROLLING_WINDOW_SETTINGS,
    "dynamic_memory_schedule_enabled": bool(RUN_DYNAMIC_MEMORY_SCHEDULE),
    "dynamic_memory_schedule": ROLLING_DYNAMIC_MEMORY_SCHEDULE,
    "head_sensitivity_enabled": bool(RUN_ROLLING_HEAD_SENSITIVITY),
    "head_sensitivity_cutoffs": ROLLING_HEAD_SENSITIVITY_CUTOFFS,
    "head_sensitivity_heads": ROLLING_HEAD_SENSITIVITY_HEADS,
    "lab_level": ROLLING_LAB_LEVEL,
    "model_head": "main run uses ridge logistic regression; optional selected-block head sensitivity includes elastic net and calibrated tree heads when available",
    "cross_validation": f"{N_SPLITS}-fold stratified patient-level CV; same folds reused across cutoffs",
    "output_dir": str(OUT_DIR),
    "private_outputs": {
        "patient_level_predictions_saved": bool(SAVE_PATIENT_LEVEL_ROLLING_PREDICTIONS),
        "note": "Patient-level rolling predictions contain person_id and should not be committed to public GitHub.",
    },
    "interpretation": [
        "Each prediction uses only events available up to the cutoff day.",
        "Risk trajectories are out-of-fold predictions, not in-sample fitted probabilities.",
        "V4 adds a prespecified horizon-aware dynamic memory schedule; lag/tau are not selected using held-out test performance.",
        "This notebook evaluates dynamic risk updating; it does not prove causal pathways.",
    ],
}
with open(OUT_DIR / "rolling_vvuq_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
manifest


## Interpretation guide

Use this notebook to test whether horizon-aware memory improves rolling recurrence-risk surveillance while preserving strict temporal truncation.

Key comparisons:

1. **Short-memory reference:** condition→condition lag 7/tau 3 + condition→lab-axis lag 2/tau 1.
2. **Medium-memory sensitivity:** lag 14/tau 7 for both transition families.
3. **Long-memory sensitivity:** lag 45/tau 14 for both transition families.
4. **Legacy-long sensitivity:** lag 90/tau 30, included only to understand whether older benchmark performance came from broad memory windows.
5. **Dynamic-memory schedule:** earlier landmarks use longer memory; later landmarks emphasize short-range condition→lab physiology. This is the V3 primary surveillance block.
6. **Prediction-head sensitivity:** selected blocks at Day -30, Day -14, Day -7, and Day 0 are re-fit with elastic net and calibrated tree heads when available.

The primary validation rule remains unchanged: at each cutoff, features are rebuilt only from events visible up to that cutoff. Longer lag/tau windows do **not** look past the cutoff; they only allow older visible events to contribute more broadly within the pre-cutoff history.

Interpret the dynamic schedule as a prespecified memory profile, not as online hyperparameter tuning on the held-out patient.

